In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../datasets/book_reviews_500.csv")

### Lowercase tatget feature

In [28]:
df["clean_text"] = df["ReviewText"].str.lower()
df.tail()

,id,BookTitle,Author,Year,GivenRating,ReviewText,ReviewDate,clean_text
495,496,Stone Soup,Jon J. Muth,2003,4.2,This book was AMAZING. [skip this part] 'Sto...,2023-12-20,this book was amazing. [skip this part] 'sto...
496,497,A Light in the Attic,Shel Silverstein,1981,3.4,This book was LOVED IT. [Review] 'A Light in...,2023-04-12,this book was loved it. [review] 'a light in...
497,498,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling,1999,4.6,This book was MAGICAL. [note] 'Harry Potter ...,2025-05-08,this book was magical. [note] 'harry potter ...
498,499,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,1998,4.4,This book was MAGICAL. [spoiler] 'Harry Pott...,2023-11-08,this book was magical. [spoiler] 'harry pott...
499,500,The Three Billy Goats Gruff,Paul Galdone,1973,4.2,This book was SO BAD. [note] 'The Three Bill...,2023-10-23,this book was so bad. [note] 'the three bill...


### Remove emojies, special chars and extra spaces etc.

In [29]:
df["clean_text"] = df["clean_text"].str.replace(r"[^a-z\s]", "", regex=True)
df["clean_text"] = df["clean_text"].str.replace(r"\s+", " ", regex=True).str.strip()


df.sample(5)

,id,BookTitle,Author,Year,GivenRating,ReviewText,ReviewDate,clean_text
416,417,Curious George,H.A. Rey,1941,5.0,This book was SO BAD. [spoiler] 'Curious Geo...,2023-06-02,this book was so bad spoiler curious george by...
267,268,"The Lion, the Witch and the Wardrobe (Chronicl...",C.S. Lewis,1950,4.2,"This book was AMAZING. [spoiler] 'The Lion, ...",2024-03-03,this book was amazing spoiler the lion the wit...
210,211,Corduroy,Don Freeman,1968,4.3,This book was AMAZING. [Review] 'Corduroy' b...,2023-06-12,this book was amazing review corduroy by don f...
51,52,The Polar Express,Chris Van Allsburg,1985,3.3,This book was AMAZING. [spoiler] 'The Polar ...,2024-11-29,this book was amazing spoiler the polar expres...
225,226,The Tale of Peter Rabbit (World of Beatrix Pot...,Beatrix Potter,1902,4.2,This book was SO BAD. [Review] 'The Tale of ...,2025-07-08,this book was so bad review the tale of peter ...


### now instead of doing these all steps seperatly, a function need to be created and pass the target column

In [42]:
def clean_feature(text):
    output = text.str.lower()
    output = output.str.replace(r"[^a-z\s]", "", regex=True)
    output = output.str.replace(r"\s+", " ", regex=True).str.strip()
    output = output.fillna("")
    return output

df["clean_text"] = clean_feature(df["ReviewText"])

df.sample(5)

,id,BookTitle,Author,Year,GivenRating,ReviewText,ReviewDate,clean_text
345,346,Stellaluna,Janell Cannon,1993,3.3,This book was AMAZING. [Review] 'Stellaluna'...,2025-03-18,this book was amazing review stellaluna by jan...
179,180,From the Mixed-Up Files of Mrs. Basil E. Frank...,E.L. Konigsburg,1967,5.0,This book was BORING. [note] 'From the Mixed...,2023-08-02,this book was boring note from the mixedup fil...
106,107,"The Cat in the Hat (Cat in the Hat, #1)",Dr. Seuss,1957,5.0,This book was LOVED IT. [Review] 'The Cat in...,2024-05-24,this book was loved it review the cat in the h...
322,323,"Alexander and the Terrible, Horrible, No Good,...",Judith Viorst,1972,4.2,This book was SO BAD. [note] 'Alexander and ...,2025-08-05,this book was so bad note alexander and the te...
440,441,"Amelia Bedelia (Amelia Bedelia, #1)",Peggy Parish,1963,4.2,This book was SO BAD. [Review] 'Amelia Bedel...,2024-01-22,this book was so bad review amelia bedelia ame...


### Now we need to perform tokenization, lemmitization, removing stopwords using spacy

In [43]:
import spacy
nlp = spacy.load("en_core_web_sm")

### Tokenize one recorn first

In [44]:
des = df["clean_text"][0]

doc = nlp(des)

[token.lemma_ for token in doc if not token.is_stop] #this token is the each token and if we add lemma_ this will lamitize the token

['book', 'boring', 'review', 'wild', 'thing', 'maurice', 'sendak', 'friend']

### Creatte a function to do the process

In [45]:
def tokenize_lemmitize_remove_stopwords(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc if not token.is_stop])  

In [46]:
df["clean_text"] = df["clean_text"].apply(tokenize_lemmitize_remove_stopwords)

df.sample(5)

,id,BookTitle,Author,Year,GivenRating,ReviewText,ReviewDate,clean_text
149,150,"A Wrinkle in Time (Time Quintet, #1)",Madeleine L'Engle,1962,5.0,This book was AMAZING. [note] 'A Wrinkle in ...,2023-02-09,book amazing note wrinkle time time quintet ma...
297,298,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling,1999,4.6,This book was LOVED IT. [skip this part] 'Ha...,2025-04-03,book love skip harry potter prisoner azkaban h...
86,87,"The Little House Collection (Little House, #1-9)",Laura Ingalls Wilder,1932,4.4,This book was BORING. [note] 'The Little Hou...,2023-01-14,book boring note little house collection littl...
208,209,If You Give a Mouse a Cookie,Laura Joffe Numeroff,1985,4.3,This book was LOVED IT. [spoiler] 'If You Gi...,2025-05-02,book love spoiler mouse cookie laura joffe num...
179,180,From the Mixed-Up Files of Mrs. Basil E. Frank...,E.L. Konigsburg,1967,5.0,This book was BORING. [note] 'From the Mixed...,2023-08-02,book boring note mixedup file mrs basil e fran...


### Vectorization

In [52]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(stop_words="english") ## pass stop_words to remove stop words(we already doen using spacy still we can do)

dtm = cv.fit_transform(df["clean_text"])

In [54]:
#to check the feature columns
new_df = dtm.toarray()
# new_df
cv.get_feature_names_out()

#make a pandas dataframe to view the colom wise vectors
new_df1 = pd.DataFrame(new_df, columns=cv.get_feature_names_out())
new_df1

,aa,adventure,alexander,alice,allard,allsburg,amazing,amelia,anne,antoine,...,winniethepooh,wise,witch,wizard,wonderful,wonderland,wood,world,wrinkle,zion
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
496,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
497,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
498,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### We can also use TfIdf for vectorization

In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(stop_words="english") ## pass stop_words to remove stop words(we already doen using spacy still we can do)

dtm_tf = tf.fit_transform(df["clean_text"])

In [60]:
new_tf = dtm_tf.toarray()
new_tf
tf.get_feature_names_out()

#make a pandas dataframe to view the colom wise vectors
new_tf1 = pd.DataFrame(new_tf, columns=tf.get_feature_names_out())
new_tf1

,aa,adventure,alexander,alice,allard,allsburg,amazing,amelia,anne,antoine,...,winniethepooh,wise,witch,wizard,wonderful,wonderland,wood,world,wrinkle,zion
0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.400276,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.0,0.0,0.0,0.0,0.0,0.0,0.246522,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
496,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
497,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
498,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [62]:
#Basicaly the final python code will be this
df = pd.read_csv("../datasets/book_reviews_500.csv")
nlp = spacy.load("en_core_web_sm")

def clean_feature(text):
    output = text.str.lower()
    output = output.str.replace(r"[^a-z\s]", "", regex=True)
    output = output.str.replace(r"\s+", " ", regex=True).str.strip()
    output = output.fillna("")
    return output



def tokenize_lemmitize_remove_stopwords(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc if not token.is_stop]) 

def text_preprocessing(text):
     output = clean_feature(text)
     output = output.apply(tokenize_lemmitize_remove_stopwords)
     return output


df["fine_text"] = text_preprocessing(df["ReviewText"])

df.sample(5)


,id,BookTitle,Author,Year,GivenRating,ReviewText,ReviewDate,fine_text
38,39,The Little Engine That Could,Watty Piper,1930,5.0,This book was SO BAD. [note] 'The Little Eng...,2025-01-10,book bad note little engine watty piper httpbu...
352,353,The Phantom Tollbooth,Norton Juster,1961,4.2,This book was BORING. [Review] 'The Phantom ...,2025-02-25,book boring review phantom tollbooth norton ju...
445,446,Stellaluna,Janell Cannon,1993,5.0,This book was SO BAD. [note] 'Stellaluna' by...,2024-12-07,book bad note stellaluna janell cannon friend
231,232,The Secret Garden,Frances Hodgson Burnett,1911,3.2,This book was MAGICAL. [Review] 'The Secret ...,2024-05-21,book magical review secret garden frances hodg...
394,395,Bunnicula: A Rabbit-Tale of Mystery (Bunnicula...,Deborah Howe,1979,3.9,This book was LOVED IT. [Review] 'Bunnicula:...,2023-01-18,book love review bunnicula rabbittale mystery ...
